In [1]:
# Imports Only
from scipy import stats
import numpy as np

# GAP vs Flattening

In [2]:
def compute_stats(flatten_scores,gap_scores):
    print(f"Flatten Avg: {np.mean(flatten_scores):.3f} | GAP Avg: {np.mean(gap_scores):.3f}")
    print(f"Flatten Std: {np.std(flatten_scores, ddof=1):.3f} | GAP Std: {np.std(gap_scores, ddof=1):.3f}")

    t_stat, p_value = stats.ttest_rel(flatten_scores, gap_scores)  # paired
    print(f"p-value: {p_value:.4f}")
    print(f"Significant: {p_value < 0.05}")

    # Confidence interval on the difference
    diff = np.array(gap_scores) - np.array(flatten_scores)
    ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
    print(f"Δ: {diff.mean():.3f}")
    print(f"95% CI on Δ: [{ci[0]:.3f}, {ci[1]:.3f}]")

In [3]:
# --- Load flatten vs GAP test scores from training logs --------------------
# Each log has a "Saved test metrics to .../test_metrics.json" line that links
# it to its metrics dir (hash-prefixed). That JSON holds the per-seed `metrics`
# (roc_auc, f1, pr_auc) and the `seed`. We pair flatten vs GAP by seed.
#
# The "default" variant (no flat/gap token in the log name) differs per model:
#   cnn, resnet        -> default == FLATTEN (only a *_gap_*  variant exists)
#   omnirad, radiodino -> default == GAP     (only a *_flat_* variant exists)
# Threshold 7.0 only (thr6 logs excluded). Seeds 42-46 map to runs 1-5.
import re, json, os

LOGS_DIR = "../results/logs"
SEEDS = [42, 43, 44, 45, 46]
RUN_OF_SEED = {42: 1, 43: 2, 44: 3, 45: 4, 46: 5}

FLAT_GAP_LOGS = {
    "cnn":       {"flatten": "cnn_v0_seed{s}_run{r}",          "gap": "cnn_v0_gap_seed{s}_run{r}"},
    "resnet":    {"flatten": "resnet_v0_seed{s}_run{r}",       "gap": "resnet_v0_gap_seed{s}_run{r}"},
    "omnirad":   {"flatten": "omnirad_v5_flat_seed{s}_run{r}", "gap": "omnirad_v5_seed{s}_run{r}"},
    "radiodino": {"flatten": "radiodino_v5_flat_seed{s}_run{r}", "gap": "radiodino_v5_seed{s}_run{r}"},
}

_SAVED_RE = re.compile(r"Saved test metrics to (\S+test_metrics\.json)")

def _metrics_path_from_log(log_name):
    log_path = os.path.join(LOGS_DIR, log_name + ".log")
    with open(log_path, encoding="utf-8", errors="replace") as fh:
        m = _SAVED_RE.search(fh.read())
    if not m:
        raise ValueError(f"No 'Saved test metrics to' line in {log_path}")
    return m.group(1)

def _load_scores(log_pattern, metric):
    """Metric values across SEEDS, ordered by seed (so flat/gap stay paired)."""
    by_seed = {}
    for s in SEEDS:
        log_name = log_pattern.format(s=s, r=RUN_OF_SEED[s])
        with open(_metrics_path_from_log(log_name)) as fh:
            d = json.load(fh)
        by_seed[d["seed"]] = d["metrics"][metric]
    return [by_seed[s] for s in SEEDS]

def flat_gap(model, metric):
    """Return (flatten_scores, gap_scores) paired by seed for a model/metric."""
    pats = FLAT_GAP_LOGS[model]
    return _load_scores(pats["flatten"], metric), _load_scores(pats["gap"], metric)

## 3D CNN

In [4]:
# ROC AUC
flatten_scores, gap_scores = flat_gap("cnn", "roc_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.829 | GAP Avg: 0.830
Flatten Std: 0.038 | GAP Std: 0.033
p-value: 0.9131
Significant: False
Δ: 0.002
95% CI on Δ: [-0.038, 0.041]


In [5]:
# F1 score
flatten_scores, gap_scores = flat_gap("cnn", "f1")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.660 | GAP Avg: 0.678
Flatten Std: 0.041 | GAP Std: 0.041
p-value: 0.4182
Significant: False
Δ: 0.018
95% CI on Δ: [-0.037, 0.073]


In [6]:
# PR AUC
flatten_scores, gap_scores = flat_gap("cnn", "pr_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.720 | GAP Avg: 0.709
Flatten Std: 0.071 | GAP Std: 0.066
p-value: 0.6705
Significant: False
Δ: -0.011
95% CI on Δ: [-0.078, 0.056]


## 3D ResNet

In [7]:
# ROC AUC
flatten_scores, gap_scores = flat_gap("resnet", "roc_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.812 | GAP Avg: 0.811
Flatten Std: 0.045 | GAP Std: 0.046
p-value: 0.8888
Significant: False
Δ: -0.001
95% CI on Δ: [-0.025, 0.023]


In [8]:
# F1 score
flatten_scores, gap_scores = flat_gap("resnet", "f1")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.629 | GAP Avg: 0.631
Flatten Std: 0.060 | GAP Std: 0.055
p-value: 0.9461
Significant: False
Δ: 0.001
95% CI on Δ: [-0.054, 0.057]


In [9]:
# PR AUC
flatten_scores, gap_scores = flat_gap("resnet", "pr_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.682 | GAP Avg: 0.683
Flatten Std: 0.077 | GAP Std: 0.079
p-value: 0.9418
Significant: False
Δ: 0.001
95% CI on Δ: [-0.032, 0.034]


## OmniRad

In [10]:
# ROC AUC
flatten_scores, gap_scores = flat_gap("omnirad", "roc_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.802 | GAP Avg: 0.786
Flatten Std: 0.045 | GAP Std: 0.043
p-value: 0.2460
Significant: False
Δ: -0.016
95% CI on Δ: [-0.049, 0.017]


In [11]:
# F1 score
flatten_scores, gap_scores = flat_gap("omnirad", "f1")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.618 | GAP Avg: 0.611
Flatten Std: 0.041 | GAP Std: 0.034
p-value: 0.6690
Significant: False
Δ: -0.007
95% CI on Δ: [-0.050, 0.036]


In [12]:
# PR AUC
flatten_scores, gap_scores = flat_gap("omnirad", "pr_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.699 | GAP Avg: 0.640
Flatten Std: 0.032 | GAP Std: 0.083
p-value: 0.0601
Significant: False
Δ: -0.059
95% CI on Δ: [-0.122, 0.004]


## RadioDino

In [13]:
# ROC AUC
flatten_scores, gap_scores = flat_gap("radiodino", "roc_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.821 | GAP Avg: 0.801
Flatten Std: 0.029 | GAP Std: 0.050
p-value: 0.1267
Significant: False
Δ: -0.020
95% CI on Δ: [-0.049, 0.009]


In [14]:
# F1 score
flatten_scores, gap_scores = flat_gap("radiodino", "f1")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.657 | GAP Avg: 0.635
Flatten Std: 0.038 | GAP Std: 0.046
p-value: 0.2556
Significant: False
Δ: -0.022
95% CI on Δ: [-0.068, 0.024]


In [15]:
# PR AUC
flatten_scores, gap_scores = flat_gap("radiodino", "pr_auc")
compute_stats(flatten_scores, gap_scores)

Flatten Avg: 0.706 | GAP Avg: 0.686
Flatten Std: 0.057 | GAP Std: 0.071
p-value: 0.2140
Significant: False
Δ: -0.020
95% CI on Δ: [-0.059, 0.018]


# CLS vs Dense Features

In [16]:
def compute_stats(scores_one,scores_two,labels: list):
    print(f"{labels[0]} Avg: {np.mean(scores_one):.3f} | {labels[1]} Avg: {np.mean(scores_two):.3f}")
    print(f"{labels[0]} Std: {np.std(scores_one, ddof=1):.3f} | {labels[1]} Std: {np.std(scores_two, ddof=1):.3f}")

    t_stat, p_value = stats.ttest_rel(scores_one, scores_two)  # paired
    print(f"p-value: {p_value:.4f}")
    print(f"Significant: {p_value < 0.05}")

    # Confidence interval on the difference
    diff = np.array(scores_two) - np.array(scores_one)
    ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
    print(f"Δ: {diff.mean():.3f}")
    print(f"95% CI on Δ: [{ci[0]:.3f}, {ci[1]:.3f}]")

In [17]:
# --- Load CLS-token vs Dense-feature test scores ---------------------------
# Only OmniRad / RadioDino have a CLS token (the CNN/ResNet do not). CLS runs
# carry a "cls" token in the log name; Dense = the same setup without it.
# We use the FLATTEN dense runs as the Dense baseline (flatten beat GAP on
# average above). Paired by seed, threshold 7.0, seeds 42-46 -> runs 1-5.
CLS_DENSE_LOGS = {
    "omnirad":   {"cls": "omnirad_v5_cls_seed{s}_run{r}",   "dense": "omnirad_v5_flat_seed{s}_run{r}"},
    "radiodino": {"cls": "radiodino_v5_cls_seed{s}_run{r}", "dense": "radiodino_v5_flat_seed{s}_run{r}"},
}

def cls_dense(model, metric):
    """Return (cls_scores, dense_scores) paired by seed for a model/metric."""
    pats = CLS_DENSE_LOGS[model]
    return _load_scores(pats["cls"], metric), _load_scores(pats["dense"], metric)

Δ is reported as **Dense − CLS** (positive favours Dense).

## OmniRad

In [18]:
# ROC AUC
cls_scores, dense_scores = cls_dense("omnirad", "roc_auc")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.776 | Dense Avg: 0.802
CLS Std: 0.038 | Dense Std: 0.045
p-value: 0.0682
Significant: False
Δ: 0.026
95% CI on Δ: [-0.003, 0.056]


In [19]:
# F1 score
cls_scores, dense_scores = cls_dense("omnirad", "f1")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.611 | Dense Avg: 0.618
CLS Std: 0.047 | Dense Std: 0.041
p-value: 0.7709
Significant: False
Δ: 0.007
95% CI on Δ: [-0.055, 0.069]


In [20]:
# PR AUC
cls_scores, dense_scores = cls_dense("omnirad", "pr_auc")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.642 | Dense Avg: 0.699
CLS Std: 0.029 | Dense Std: 0.032
p-value: 0.0085
Significant: True
Δ: 0.057
95% CI on Δ: [0.024, 0.090]


## RadioDino

In [21]:
# ROC AUC
cls_scores, dense_scores = cls_dense("radiodino", "roc_auc")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.807 | Dense Avg: 0.821
CLS Std: 0.040 | Dense Std: 0.029
p-value: 0.1530
Significant: False
Δ: 0.014
95% CI on Δ: [-0.008, 0.036]


In [22]:
# F1 score
cls_scores, dense_scores = cls_dense("radiodino", "f1")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.643 | Dense Avg: 0.657
CLS Std: 0.051 | Dense Std: 0.038
p-value: 0.5574
Significant: False
Δ: 0.014
95% CI on Δ: [-0.047, 0.075]


In [23]:
# PR AUC
cls_scores, dense_scores = cls_dense("radiodino", "pr_auc")
compute_stats(cls_scores, dense_scores, ["CLS", "Dense"])

CLS Avg: 0.688 | Dense Avg: 0.706
CLS Std: 0.056 | Dense Std: 0.057
p-value: 0.0523
Significant: False
Δ: 0.018
95% CI on Δ: [-0.000, 0.037]


# Transfer Learning Strategies

Encoder fine-tuning depth for the ViT models (OmniRad, RadioDino), full fine-tune (v5) as the reference:

- **v0** – Transfer learning (frozen encoder, head only)
- **v2** – Partial fine-tune (last-N blocks + embed)
- **v5** – Full fine-tune (all layers)

Uses the **flat** aggregation runs (flatten beat GAP above) so aggregation is held constant. CNN/ResNet excluded (no pretrained ViT to transfer). Paired two-sided t-tests by seed, threshold 7.0; p-values are uncorrected.

In [24]:
# --- Load v0/v2/v5 transfer-learning test scores (flat runs) ----------------
TRANSFER_LOGS = {
    "omnirad": {
        "v0": "omnirad_v0_flat_seed{s}_run{r}",
        "v2": "omnirad_v2_flat_seed{s}_run{r}",
        "v5": "omnirad_v5_flat_seed{s}_run{r}",
    },
    "radiodino": {
        "v0": "radiodino_v0_flat_seed{s}_run{r}",
        "v2": "radiodino_v2_flat_seed{s}_run{r}",
        "v5": "radiodino_v5_flat_seed{s}_run{r}",
    },
}

def transfer_scores(model, version, metric):
    """metric values across SEEDS (ordered by seed) for one finetune strategy."""
    return _load_scores(TRANSFER_LOGS[model][version], metric)

## OmniRad

### Partial Fine-Tuning

In [25]:
# ROC AUC
full_scores    = transfer_scores("omnirad", "v5", "roc_auc")
v2_scores = transfer_scores("omnirad", "v2", "roc_auc")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.802 | Partial Fine-tune Avg: 0.811
Full Fine-tune Std: 0.045 | Partial Fine-tune Std: 0.036
p-value: 0.3208
Significant: False
Δ: 0.009
95% CI on Δ: [-0.013, 0.031]


In [26]:
# F1 score
full_scores    = transfer_scores("omnirad", "v5", "f1")
v2_scores = transfer_scores("omnirad", "v2", "f1")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.618 | Partial Fine-tune Avg: 0.617
Full Fine-tune Std: 0.041 | Partial Fine-tune Std: 0.058
p-value: 0.9543
Significant: False
Δ: -0.001
95% CI on Δ: [-0.055, 0.053]


In [27]:
# PR AUC
full_scores    = transfer_scores("omnirad", "v5", "pr_auc")
v2_scores = transfer_scores("omnirad", "v2", "pr_auc")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.699 | Partial Fine-tune Avg: 0.702
Full Fine-tune Std: 0.032 | Partial Fine-tune Std: 0.050
p-value: 0.8071
Significant: False
Δ: 0.003
95% CI on Δ: [-0.030, 0.037]


### Transfer Learning

In [28]:
# ROC AUC
full_scores    = transfer_scores("omnirad", "v5", "roc_auc")
v0_scores = transfer_scores("omnirad", "v0", "roc_auc")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.802 | Transfer Learning Avg: 0.787
Full Fine-tune Std: 0.045 | Transfer Learning Std: 0.036
p-value: 0.4163
Significant: False
Δ: -0.015
95% CI on Δ: [-0.062, 0.032]


In [29]:
# F1 score
full_scores    = transfer_scores("omnirad", "v5", "f1")
v0_scores = transfer_scores("omnirad", "v0", "f1")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.618 | Transfer Learning Avg: 0.614
Full Fine-tune Std: 0.041 | Transfer Learning Std: 0.038
p-value: 0.7488
Significant: False
Δ: -0.004
95% CI on Δ: [-0.039, 0.031]


In [30]:
# PR AUC
full_scores    = transfer_scores("omnirad", "v5", "pr_auc")
v0_scores = transfer_scores("omnirad", "v0", "pr_auc")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.699 | Transfer Learning Avg: 0.655
Full Fine-tune Std: 0.032 | Transfer Learning Std: 0.052
p-value: 0.0096
Significant: True
Δ: -0.044
95% CI on Δ: [-0.070, -0.018]


## RadioDino

### Partial Fine-Tuning

In [31]:
# ROC AUC
full_scores    = transfer_scores("radiodino", "v5", "roc_auc")
v2_scores = transfer_scores("radiodino", "v2", "roc_auc")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.821 | Partial Fine-tune Avg: 0.796
Full Fine-tune Std: 0.029 | Partial Fine-tune Std: 0.034
p-value: 0.0010
Significant: True
Δ: -0.025
95% CI on Δ: [-0.033, -0.017]


In [32]:
# F1 score
full_scores    = transfer_scores("radiodino", "v5", "f1")
v2_scores = transfer_scores("radiodino", "v2", "f1")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.657 | Partial Fine-tune Avg: 0.606
Full Fine-tune Std: 0.038 | Partial Fine-tune Std: 0.025
p-value: 0.0520
Significant: False
Δ: -0.051
95% CI on Δ: [-0.103, 0.001]


In [33]:
# PR AUC
full_scores    = transfer_scores("radiodino", "v5", "pr_auc")
v2_scores = transfer_scores("radiodino", "v2", "pr_auc")
compute_stats(full_scores, v2_scores, ["Full Fine-tune", "Partial Fine-tune"])

Full Fine-tune Avg: 0.706 | Partial Fine-tune Avg: 0.667
Full Fine-tune Std: 0.057 | Partial Fine-tune Std: 0.046
p-value: 0.0086
Significant: True
Δ: -0.040
95% CI on Δ: [-0.062, -0.017]


### Transfer Learning

In [34]:
# ROC AUC
full_scores    = transfer_scores("radiodino", "v5", "roc_auc")
v0_scores = transfer_scores("radiodino", "v0", "roc_auc")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.821 | Transfer Learning Avg: 0.783
Full Fine-tune Std: 0.029 | Transfer Learning Std: 0.043
p-value: 0.0208
Significant: True
Δ: -0.038
95% CI on Δ: [-0.066, -0.009]


In [35]:
# F1 score
full_scores    = transfer_scores("radiodino", "v5", "f1")
v0_scores = transfer_scores("radiodino", "v0", "f1")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.657 | Transfer Learning Avg: 0.600
Full Fine-tune Std: 0.038 | Transfer Learning Std: 0.050
p-value: 0.0251
Significant: True
Δ: -0.058
95% CI on Δ: [-0.104, -0.012]


In [36]:
# PR AUC
full_scores    = transfer_scores("radiodino", "v5", "pr_auc")
v0_scores = transfer_scores("radiodino", "v0", "pr_auc")
compute_stats(full_scores, v0_scores, ["Full Fine-tune", "Transfer Learning"])

Full Fine-tune Avg: 0.706 | Transfer Learning Avg: 0.657
Full Fine-tune Std: 0.057 | Transfer Learning Std: 0.074
p-value: 0.0823
Significant: False
Δ: -0.049
95% CI on Δ: [-0.108, 0.010]
